In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

# Retrieval

In [3]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()


In [4]:
courses_raw

[{'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 161},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 493},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 76},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 398}]

In [5]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"{url_prefix}/{course["path"]}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1470

In [6]:
documents[10]

{'id': '20c5a1347e',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?',
 'answer': 'Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nIt contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps.'}

In [7]:
from minsearch import Index

index = Index(
    text_fields = ["question", "section", "answer"],
    keyword_fields = ["course"]
)

index.fit(documents)

In [8]:
question = "I just discovered the course. Can I still join?"
search_results = index.search(question, filter_dict={"course": "llm-zoomcamp"}, num_results=5)

In [9]:
boosted_search_results = index.search(question, 
                              boost_dict={"question": 2.0, "section": 0.5}, # question field is 2x more important
                              filter_dict={"course": "llm-zoomcamp"}, 
                              num_results=5)

In [10]:
def search(question, course="llm-zoomcamp"):
    return index.search(question, 
                        boost_dict={"question": 2.0, "section": 0.5}, 
                        filter_dict={"course": course}, 
                        num_results=5
                        )

In [11]:
search_res = search(question)

# Building Prompt

In [12]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with 'I don't know.'
"""

In [13]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [14]:
context = build_context(search_res)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expec

In [15]:
USER_PROMPT_TEMPLATE = f"""
Question:
{question}

Context:
{context}
"""

In [16]:
USER_PROMPT_TEMPLATE.format(question=question, context=context)

"\nQuestion:\nI just discovered the course. Can I still join?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: The homework submission form is still open even though the deadline has passed — can I still submit?\nA: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.\n\nGeneral Course-Related Questions\nQ: I missed the first homework - can I still get a certificate?\nA: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.\n\nGeneral Cours

In [17]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

In [18]:
prompt = build_prompt(question, search_res)

In [19]:
print(prompt)

Question:
I just discovered the course. Can I still join?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions

# Adding LLM

In [20]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)
response.output_text

'Yes — you can still join.\n\nYou don’t need a separate confirmation or registration check to start learning. You can begin the course and submit homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.'

In [21]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0589a5fd9ee15603006af0d24cab6087d29fe438f3d20d9a6d",
  "access_programs": null,
  "created_at": 1794167372.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0589a5fd9ee15603006af0d24e0d1087d29a62f91d69bef50a",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can still join.\n\nYou don’t need a separate confirmation or registration check to start learning. You can begin the course and submit homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": []

In [22]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
    model=model,
    input=user_prompt
    )

    return response.output_text

In [23]:
def rag(query, model="gpt-5.4-mini"):
    search_res = search(query)
    prompt = build_prompt(query, search_res)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [24]:
answer = rag(question)
print(answer)

Yes — you can still join.

You don’t need a separate confirmation to start learning or submitting homework. If you want a certificate, make sure you submit your project while submissions are still open.
